# Load magnitude (Sec. 4.4)

Generates `figures/load_magnitude.pdf` from `results/4_load_magnitude/`.

Each design is thresholded to its V\* = 0.5. Stress and creep strain come from the runs' `.vtu` files and are drawn on solid cells only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from plot_utils import (CBAR_LABEL_FS, CBAR_TICK_FS, FIGURES, RESULTS, TITLE_FS, equivalent_strain, label_rows,
                        load_design, plot_field, solid_bounds, specimen_cmap, von_mises_stress)

VF = 0.5


def load_case(load):
    design = load_design(RESULTS / "4_load_magnitude" / f"data_creep_1.0_{load:.1f}_0.5" / "vtk" / "sol_250.vtu", VF)
    return {
        "design": design,
        "vm_sig_0": von_mises_stress(design.mesh.cell_data["stress_0"][0]),
        "eq_eps_f": equivalent_strain(design.mesh.cell_data["creep_strain_final"][0]),
    }


def draw_bcs(ax, design):
    """Clamped left edge, with traction on the middle 20% of the right edge."""
    (xmin, ymin), (xmax, ymax) = design.points.min(axis=0), design.points.max(axis=0)
    height = ymax - ymin
    y_mid = 0.5 * (ymin + ymax)
    y_start, y_end = y_mid - 0.1 * height, y_mid + 0.1 * height
    y_supports = np.linspace(ymin, ymax, 8)
    ax.plot(np.full_like(y_supports, xmin - 3.0), y_supports, marker=">", markersize=8,
            color="red", linestyle="none", clip_on=False, zorder=5)
    ax.plot([xmax, xmax], [y_start, y_end], color="blue", linewidth=4.0, solid_capstyle="butt",
            clip_on=False, zorder=5)
    ax.annotate("", xy=(xmax + 6.0, y_start), xytext=(xmax + 6.0, y_end),
                arrowprops=dict(arrowstyle="-|>", color="blue", linewidth=2.0, joinstyle="miter"),
                annotation_clip=False, zorder=6)

In [ ]:
loads = [100.0, 200.0, 400.0]
label_kwargs = dict(fontsize=CBAR_LABEL_FS, rotation=0, labelpad=20, va="center", ha="left")

fig, axes = plt.subplots(3, 3, figsize=(16, 7))
for j, load in enumerate(loads):
    case = load_case(load)
    design = case["design"]
    last_column = j == len(loads) - 1

    # Each column has its own color scales.
    sig_lo, sig_hi = solid_bounds(design, case["vm_sig_0"])
    eps_lo, eps_hi = solid_bounds(design, case["eq_eps_f"])

    c_geom = plot_field(axes[0, j], design, design.binary, cmap=specimen_cmap, vmin=0.0, vmax=1.0)
    draw_bcs(axes[0, j], design)
    c_sig = plot_field(axes[1, j], design, case["vm_sig_0"], cmap="viridis", mask=design.void_mask,
                       vmin=sig_lo, vmax=sig_hi)
    c_eps = plot_field(axes[2, j], design, case["eq_eps_f"], cmap="magma", mask=design.void_mask,
                       vmin=eps_lo, vmax=eps_hi, log=True)
    axes[0, j].set_title(f"{load:.0f} MPa Load", fontsize=TITLE_FS)

    cb_geom = fig.colorbar(c_geom, ax=axes[0, j], shrink=0.8, pad=0.02, ticks=[0.0, 1.0])
    cb_geom.set_ticklabels(["0", "1"])
    cb_sig = fig.colorbar(c_sig, ax=axes[1, j], shrink=0.8, pad=0.02)
    cb_eps = fig.colorbar(c_eps, ax=axes[2, j], shrink=0.8, pad=0.02)
    for cb, label in ((cb_geom, r"$\rho$"), (cb_sig, r"$\sigma_{eq}$ [MPa]"), (cb_eps, r"$\varepsilon_{eq,creep}$")):
        cb.ax.tick_params(labelsize=CBAR_TICK_FS)
        if last_column:
            cb.set_label(label, **label_kwargs)

label_rows(axes[:, 0], ["Optimized \nGeometry", "Initial\n Equivalent\n Stress", "Final\n Equivalent\n Creep Strain"])
fig.subplots_adjust(wspace=0.1, hspace=0.1)

fig.savefig(FIGURES / "load_magnitude.pdf", bbox_inches="tight")
plt.show()